# 🌐 Federated Learning Training — FedAvg Simulation
**Privacy-Preserving Cancer Detection Project**

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU → Save**
2. Upload these to **MyDrive/FL_Project/** on Google Drive:
   - `ml/` folder (model.py)
   - `data/` folder (node1/, node2/, node3/, test/)
3. Run all cells **top to bottom** (Runtime → Run all)

---

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅  Drive mounted.")

## Step 2 — Check what files exist on Drive

In [ ]:
import os

base = "/content/drive/MyDrive/FL_Project"
print(f"Scanning {base} ...")
print()

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "  " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * 2 * (level + 1)
    for file in files[:5]:
        size = os.path.getsize(os.path.join(root, file)) / (1024*1024)
        print(f"{subindent}{file}  ({size:.1f} MB)")
    if len(files) > 5:
        print(f"{subindent}... and {len(files)-5} more")

## Step 3 — Copy everything from Drive to Colab VM

In [ ]:
import shutil, os

PROJECT_VM   = "/content/FL_Project"
DRIVE_PATH   = "/content/drive/MyDrive/FL_Project"

os.makedirs(PROJECT_VM, exist_ok=True)

# Copy all needed folders
for folder in ["ml", "data"]:
    src = os.path.join(DRIVE_PATH, folder)
    dst = os.path.join(PROJECT_VM, folder)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  ✅  Copied {folder}/")
    else:
        print(f"  ❌  {folder}/ NOT FOUND on Drive!")

# Create output dirs
os.makedirs(f"{PROJECT_VM}/models", exist_ok=True)
os.makedirs(f"{PROJECT_VM}/results", exist_ok=True)
print()
print("✅  All files copied to Colab VM.")

## Step 4 — Verify all required files

In [ ]:
import os, numpy as np

P = "/content/FL_Project"

required = [
    "ml/model.py",
    "data/node1/X.npy", "data/node1/y.npy",
    "data/node2/X.npy", "data/node2/y.npy",
    "data/node3/X.npy", "data/node3/y.npy",
    "data/test/X.npy",  "data/test/y.npy",
]

all_ok = True
for f in required:
    path = os.path.join(P, f)
    exists = os.path.exists(path)
    if exists and f.endswith(".npy"):
        arr = np.load(path)
        size = os.path.getsize(path) / (1024**2)
        print(f"  ✅  {f}  shape={arr.shape}  ({size:.1f} MB)")
    elif exists:
        print(f"  ✅  {f}")
    else:
        print(f"  ❌  {f}  MISSING")
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Upload missing files to MyDrive/FL_Project/")
print("
✅  All files present. Ready to train!")

## Step 5 — Run Federated Learning (10 rounds, FedAvg, 3 nodes)

This is the main training cell. It simulates 3 hospital nodes,
each training locally, then averages their weights (FedAvg).

In [ ]:
import os, sys, json, time, gc
import numpy as np
import tensorflow as tf

# ── Fix OOM: let GPU allocate memory gradually ──────────────────────
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print(f"GPUs: {len(gpus)}  (memory growth enabled)")

# ── Config ────────────────────────────────────────────────────────────
PROJECT      = "/content/FL_Project"
DATA_DIR     = f"{PROJECT}/data"
MODEL_DIR    = f"{PROJECT}/models"
RESULTS_DIR  = f"{PROJECT}/results"

NUM_ROUNDS     = 10
LOCAL_EPOCHS   = 3
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
NUM_NODES      = 3
FRACTION_TRAIN = 0.67

# ── Import model ──────────────────────────────────────────────────────
sys.path.insert(0, f"{PROJECT}/ml")
from model import build_cnn

# ── Create ONE model and reuse it (avoids OOM) ───────────────────────
tf.keras.backend.clear_session()
gc.collect()

model = build_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

# ── Helper functions (reuse single model) ────────────────────────────
def load_node(node_id):
    X = np.load(f"{DATA_DIR}/node{node_id}/X.npy")
    y = np.load(f"{DATA_DIR}/node{node_id}/y.npy")
    return X, y

def fedavg(global_w, client_results):
    """Weighted average of client weights by sample count."""
    total = sum(n for _, n in client_results)
    return [
        sum(w[i] * (n / total) for w, n in client_results)
        for i in range(len(global_w))
    ]

def train_client(node_id, global_w):
    X, y = load_node(node_id)
    model.set_weights(global_w)
    h = model.fit(X, y, epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE,
                  validation_split=0.1, verbose=0)
    return model.get_weights(), len(X), h.history["loss"][-1], h.history["accuracy"][-1]

# ── Load test data ────────────────────────────────────────────────────
X_test = np.load(f"{DATA_DIR}/test/X.npy")
y_test = np.load(f"{DATA_DIR}/test/y.npy")

# ── Initialize ────────────────────────────────────────────────────────
global_w = model.get_weights()

init_loss, init_acc = model.evaluate(X_test, y_test, verbose=0)

print()
print("═" * 60)
print("  Federated Learning — Cancer Detection")
print("  FedAvg | 10 rounds | 3 nodes | 3 local epochs")
print("═" * 60)
print(f"  Initial accuracy (random): {init_acc:.4f}")
print()

# ── FL Training Loop ──────────────────────────────────────────────────
round_metrics = []
t0 = time.time()

for r in range(1, NUM_ROUNDS + 1):
    rt = time.time()
    k = max(1, int(NUM_NODES * FRACTION_TRAIN))
    selected = sorted(np.random.choice(range(1, NUM_NODES+1), size=k, replace=False))
    print(f"Round {r:2d}/{NUM_ROUNDS}  nodes={list(selected)}  ", end="")

    results = []
    for nid in selected:
        w, n, loss, acc = train_client(nid, global_w)
        results.append((w, n))

    global_w = fedavg(global_w, results)

    # Evaluate
    model.set_weights(global_w)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    elapsed = time.time() - rt

    round_metrics.append({
        "round": r, "nodes": list(map(int, selected)),
        "test_loss": round(float(test_loss), 4),
        "test_accuracy": round(float(test_acc), 4),
        "time_sec": round(elapsed, 1)
    })
    print(f"test_acc={test_acc:.4f}  loss={test_loss:.4f}  ({elapsed:.1f}s)")

    gc.collect()

total_time = time.time() - t0
final_acc = round_metrics[-1]["test_accuracy"]
final_loss = round_metrics[-1]["test_loss"]

print()
print("═" * 60)
print(f"  ✅  FL Training Complete!")
print(f"  Final test accuracy:  {final_acc:.4f}")
print(f"  Final test loss:      {final_loss:.4f}")
print(f"  Total time:           {total_time:.1f}s")
print("═" * 60)


## Step 6 — Save federated model

In [ ]:
import os, json

# Save model (reuse the existing one)
os.makedirs(MODEL_DIR, exist_ok=True)
model.set_weights(global_w)

model_path = f"{MODEL_DIR}/federated_model.h5"
model.save(model_path)
size = os.path.getsize(model_path) / (1024**2)
print(f"✅  Model saved: {model_path}  ({size:.1f} MB)")

# Save metrics
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics = {
    "strategy": "FedAvg",
    "num_rounds": NUM_ROUNDS,
    "num_nodes": NUM_NODES,
    "local_epochs": LOCAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "initial_test_accuracy": round(float(init_acc), 4),
    "final_test_accuracy": final_acc,
    "final_test_loss": final_loss,
    "total_training_time_sec": round(total_time, 1),
    "round_metrics": round_metrics,
}
metrics_path = f"{RESULTS_DIR}/fl_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"✅  Metrics saved: {metrics_path}")


## Step 7 — Copy model + metrics back to Google Drive

In [ ]:
import shutil, os

DRIVE_OUT = "/content/drive/MyDrive/FL_Project"

saves = {
    f"{MODEL_DIR}/federated_model.h5":   f"{DRIVE_OUT}/models/federated_model.h5",
    f"{RESULTS_DIR}/fl_metrics.json":     f"{DRIVE_OUT}/results/fl_metrics.json",
}

for src, dst in saves.items():
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        size = os.path.getsize(src) / (1024**2)
        print(f"  ✅  {os.path.basename(src)}  ({size:.2f} MB) → Drive")
    else:
        print(f"  ❌  {src} not found")

print()
print("✅  Done! federated_model.h5 is on your Google Drive.")
print(f"   Location: {DRIVE_OUT}/models/")
print()
print("Next: download federated_model.h5 from Drive to your local")
print("      models/ folder to complete the project.")

## Step 8 — Display round-by-round metrics

In [ ]:
import json

print("═" * 55)
print("  Round-by-Round FL Metrics")
print("═" * 55)
print(f"  {"Round":<7} {"Nodes":<12} {"Test Acc":<12} {"Test Loss":<12} {"Time"}")
print("─" * 55)
for rm in round_metrics:
    print(f"  {rm["round"]:<7} {str(rm["nodes"]):<12} {rm["test_accuracy"]:<12.4f} {rm["test_loss"]:<12.4f} {rm["time_sec"]}s")
print("─" * 55)
print(f"  Final accuracy: {final_acc:.4f}")
print(f"  Total time:     {total_time:.1f}s")